# GNN-BERT Music Context Understanding — End-to-End Demo

This notebook is the submission's required `demo_context.ipynb`: **one end-to-end inference example**
per Section 10 of the project spec.

It runs entirely on the **synthetic** data generator (`src/synthetic_data.py`) so it works before
you've downloaded FMA / MagnaTagATune / MusicCaps / DEAM — flip `config['data']['mode']` to `'real'`
once those are on disk (see `README.md`).

Pipeline covered here: **audio -> segments -> graph -> GNN -> BERT -> fusion -> tags/emotion**,
plus a Task-4 contrastive retrieval query.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch

from src.utils import load_config, set_seed, get_device
from src.synthetic_data import generate_synthetic_dataset, GENRE_NAMES, build_top_k_tag_vocab, tags_to_multihot
from src.graph_builder import build_graph_for_track
from src.bert_encoder import BertTagClassifier
from src.gnn_model import GNNGenreClassifier
from src.fusion_model import GNNBertFusionModel
from src.contrastive import ContrastiveGNNBert

cfg = load_config('../config.yaml')
set_seed(cfg['project']['seed'])
device = get_device('cpu')  # CPU is plenty for this small demo
print('device:', device)


## 1. Generate a handful of synthetic tracks and inspect one

In [ ]:
tracks = generate_synthetic_dataset(n_tracks=32, n_genres=8, seed=7)
tag_vocab = build_top_k_tag_vocab(tracks, k=20)

track = tracks[0]
print('track_id  :', track.track_id)
print('genre     :', track.genre)
print('tags      :', track.tags)
print('caption   :', track.caption)
print('valence   :', round(track.valence, 2), ' arousal:', round(track.arousal, 2))
print('n segments:', len(track.segments))


## 2. Build the music structure graph

We build both graph types described in Section 3: a **chord-transition graph** and a
**segment-similarity graph**. Task 2/3/4 use the segment graph by default (`config.yaml`).

In [ ]:
chord_graph = build_graph_for_track(track.segments, graph_type='chord')
segment_graph = build_graph_for_track(track.segments, graph_type='segment')

print('Chord graph  : nodes =', chord_graph.num_nodes, ' edges =', chord_graph.num_edges, ' labels =', chord_graph.node_labels)
print('Segment graph: nodes =', segment_graph.num_nodes, ' edges =', segment_graph.num_edges)


## 3. Task 1 — BERT tag classifier (inference with random-init weights)

For a fast demo we skip fine-tuning here (see `train.py --task 1` for the full training loop) and
just show the forward pass shapes with the pretrained-but-not-fine-tuned encoder.

In [ ]:
bert_model = BertTagClassifier(num_tags=len(tag_vocab), model_name=cfg['task1_bert']['model_name']).to(device)
bert_model.eval()

enc = bert_model.text_encoder.tokenize([track.caption], device=device)
with torch.no_grad():
    tag_probs = bert_model.predict_proba(enc['input_ids'], enc['attention_mask'])

top5 = torch.topk(tag_probs[0], k=5)
print('Top-5 predicted tags (untrained BERT, illustrative only):')
for idx, p in zip(top5.indices.tolist(), top5.values.tolist()):
    print(f'  {tag_vocab[idx]:15s} {p:.3f}')


## 4. Task 2 — GNN forward pass on the segment graph

A single track's graph through the GraphSAGE encoder + mean-pool readout (Algorithm 2).

In [ ]:
from torch_geometric.data import Batch

pyg_graph = segment_graph.to_pyg_data()
batch = Batch.from_data_list([pyg_graph])

gnn_model = GNNGenreClassifier(
    in_channels=segment_graph.node_features.shape[1],
    num_classes=len(GENRE_NAMES),
).to(device)
gnn_model.eval()

with torch.no_grad():
    logits = gnn_model(batch.x.to(device), batch.edge_index.to(device), batch.batch.to(device))
    genre_probs = torch.softmax(logits, dim=-1)[0]

pred_genre = GENRE_NAMES[genre_probs.argmax().item()]
print('True genre:', track.genre, ' | Predicted (untrained GNN, illustrative):', pred_genre)


## 5. Task 3 — full GNN-BERT fusion: one end-to-end inference call

This is the pipeline's centerpiece: graph + caption in, multi-label tags + valence/arousal out,
via cross-attention fusion (Algorithm 3).

In [ ]:
fusion_model = GNNBertFusionModel(
    num_tags=len(tag_vocab),
    gnn_in_channels=segment_graph.node_features.shape[1],
    bert_model_name=cfg['preprocessing']['bert_model_name'],
    fusion_type='cross_attention',
).to(device)
fusion_model.eval()

enc = fusion_model.text_encoder.tokenize([track.caption], device=device)
with torch.no_grad():
    out = fusion_model(
        batch.x.to(device), batch.edge_index.to(device), batch.batch.to(device),
        enc['input_ids'], enc['attention_mask'],
    )

tag_probs = torch.sigmoid(out['tag_logits'])[0]
top5 = torch.topk(tag_probs, k=5)
print('Track       :', track.track_id, '|', track.caption)
print('True tags   :', track.tags)
print('Top-5 (untrained fusion model, illustrative):')
for idx, p in zip(top5.indices.tolist(), top5.values.tolist()):
    print(f'  {tag_vocab[idx]:15s} {p:.3f}')

if out['valence_arousal'] is not None:
    v, a = out['valence_arousal'][0].tolist()
    print(f"Predicted valence/arousal (untrained): {v:.2f} / {a:.2f}  (true: {track.valence:.2f} / {track.arousal:.2f})")


## 6. Task 4 — contrastive retrieval: caption -> best-matching track

With a trained `ContrastiveGNNBert` (see `train.py --task 4`), retrieval works by encoding every
candidate track's graph and every query caption into the same space and ranking by cosine similarity.
Here we show the mechanics with a small in-memory candidate pool.

In [ ]:
contrastive_model = ContrastiveGNNBert(
    gnn_in_channels=segment_graph.node_features.shape[1],
    bert_model_name=cfg['preprocessing']['bert_model_name'],
).to(device)
contrastive_model.eval()

candidate_tracks = tracks[:8]
candidate_graphs = [build_graph_for_track(t.segments, graph_type='segment').to_pyg_data() for t in candidate_tracks]
candidate_batch = Batch.from_data_list(candidate_graphs)

with torch.no_grad():
    audio_emb = contrastive_model.encode_audio(candidate_batch.x.to(device), candidate_batch.edge_index.to(device), candidate_batch.batch.to(device))

query_caption = 'An energetic electronic track with driving synth and drums.'
enc_q = contrastive_model.text_encoder.tokenize([query_caption], device=device)
with torch.no_grad():
    query_emb = contrastive_model.encode_text(enc_q['input_ids'], enc_q['attention_mask'])

sims = (query_emb @ audio_emb.T)[0]
top3 = torch.topk(sims, k=3)
print('Query:', query_caption)
print('Top-3 matches (untrained contrastive model, illustrative):')
for idx, s in zip(top3.indices.tolist(), top3.values.tolist()):
    t = candidate_tracks[idx]
    print(f'  {t.track_id:18s} sim={s:.3f}  genre={t.genre:10s} caption="{t.caption}"')


## Next steps

- Run `python train.py --task 1` (and 2, 3-ablations, 4) to actually fine-tune every module above
  instead of using random-init weights — this notebook only demonstrates *shapes and wiring*.
- Switch `config.yaml`'s `data.mode` to `real` once FMA / MagnaTagATune / MusicCaps / DEAM are
  downloaded into `data/raw/` (see `README.md` for the exact links and layout).
- Run `python evaluate.py` after training to produce the final comparison table.
